In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [4]:
import torch
import pandas as pd
from src.neural_nets.models import VelocityField, FlowMapNetwork
from src.utils.maths import check_identity_property, check_semigroup_property, check_ode_consistency

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEACHER_PATH = "../checkpoints/velocity_teacher_linear.pth" 
STUDENT_PATH = "../checkpoints/flow_map_student.pth"

print(f"🔍 Lancement de la validation mathématique sur {device}...")

🔍 Lancement de la validation mathématique sur cpu...


# LINEAR INTERPOLANT

In [11]:
print("Loading models...")
teacher = VelocityField(data_dim=2, hidden_dim=128, time_dim=64).to(device)
teacher.load_state_dict(torch.load(TEACHER_PATH, map_location=device))
teacher.eval()

student = FlowMapNetwork(data_dim=2, hidden_dim=128, time_dim=64).to(device)
student.load_state_dict(torch.load(STUDENT_PATH, map_location=device))
student.eval()

Loading models...


FlowMapNetwork(
  (time_embed): TimeEmbedding()
  (net): Sequential(
    (0): Linear(in_features=130, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): SiLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): SiLU()
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [7]:
x_test = torch.randn(2000, 2).to(device)

In [13]:
print("Computing metrics...")
id_error = check_identity_property(student, x_test)

Computing metrics...


In [14]:
semigroup_error = check_semigroup_property(student, x_test)

In [15]:
ode_error = check_ode_consistency(student, teacher, x_test)

In [16]:
results = {
    "Metric": ["Identity Error (t->t)", "Semigroup Error (Composition)", "ODE Consistency (Student vs Solver)"],
    "MSE Value": [id_error, semigroup_error, ode_error],
    "Status": ["✅ Excellent" if id_error < 1e-6 else "⚠️ Check architecture",
               "✅ Good" if semigroup_error < 1e-2 else "⚠️ Training issue",
               "✅ Good" if ode_error < 1e-2 else "⚠️ Distillation issue"]
}

In [17]:
df = pd.DataFrame(results)
print("\n" + "="*50)
print("FINAL VALIDATION REPORT")
print("="*50)
print(df.to_string(index=False))
print("="*50)


FINAL VALIDATION REPORT
                             Metric  MSE Value      Status
              Identity Error (t->t)   0.000000 ✅ Excellent
      Semigroup Error (Composition)   0.006789      ✅ Good
ODE Consistency (Student vs Solver)   0.008650      ✅ Good


# STOCHASTIC INTERPOLANT

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEACHER_PATH = "../checkpoints/velocity_teacher_stochastic.pth" 
STUDENT_PATH = "../checkpoints/flow_map_student_stochastic_teacher.pth"

print(f"🔍 Lancement de la validation mathématique sur {device}...")

🔍 Lancement de la validation mathématique sur cpu...


In [9]:
print("Loading models...")
teacher = VelocityField(data_dim=2, hidden_dim=128, time_dim=64).to(device)
teacher.load_state_dict(torch.load(TEACHER_PATH, map_location=device))
teacher.eval()

student = FlowMapNetwork(data_dim=2, hidden_dim=128, time_dim=64).to(device)
student.load_state_dict(torch.load(STUDENT_PATH, map_location=device))
student.eval()

Loading models...


FlowMapNetwork(
  (time_embed): TimeEmbedding()
  (net): Sequential(
    (0): Linear(in_features=130, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): SiLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): SiLU()
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [10]:
print("Computing metrics...")
id_error = check_identity_property(student, x_test)

Computing metrics...


In [11]:
semigroup_error = check_semigroup_property(student, x_test)

In [12]:
ode_error = check_ode_consistency(student, teacher, x_test)

In [13]:
results = {
    "Metric": ["Identity Error (t->t)", "Semigroup Error (Composition)", "ODE Consistency (Student vs Solver)"],
    "MSE Value": [id_error, semigroup_error, ode_error],
    "Status": ["✅ Excellent" if id_error < 1e-6 else "⚠️ Check architecture",
               "✅ Good" if semigroup_error < 1e-2 else "⚠️ Training issue",
               "✅ Good" if ode_error < 1e-2 else "⚠️ Distillation issue"]
}

In [14]:
df = pd.DataFrame(results)
print("\n" + "="*50)
print("FINAL VALIDATION REPORT")
print("="*50)
print(df.to_string(index=False))
print("="*50)


FINAL VALIDATION REPORT
                             Metric  MSE Value      Status
              Identity Error (t->t)   0.000000 ✅ Excellent
      Semigroup Error (Composition)   0.002631      ✅ Good
ODE Consistency (Student vs Solver)   0.003899      ✅ Good
